# Spark Programming - Hands-On
Companion notebook for **Spark & Databricks** (Chapters 4 & 5).

Datasets used:
- `customers`
- `repayments`
- `transactions`

Update the `catalog` and `schema` widgets below to point at your environment before running.

In [0]:
dbutils.widgets.text("catalog", "tesco_bank_training", "Catalog")
dbutils.widgets.text("schema", "datasets", "Schema")
dbutils.widgets.text("write_schema", "", "write_schema ( Your name ie. jack_gibb)")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
write_schema = dbutils.widgets.get("schema")

customers_table = f"{catalog}.{schema}.customers"
repayments_table = f"{catalog}.{schema}.repayments"
transactions_table = f"{catalog}.{schema}.transactions"

write_schema = dbutils.widgets.get("write_schema")

write_schema = write_schema + "_" + "silver"

print(customers_table)
print(repayments_table)
print(transactions_table)
print(write_schema)

## 1. Read data into a Spark DataFrame
Two equivalent ways to read a Unity Catalog table: the PySpark API and the Spark SQL API.

In [0]:
# PySpark API - spark.table() method
customers = spark.table(customers_table)
display(customers)

In [0]:
# Spark SQL API - spark.sql() method
transactions = spark.sql(f"""SELECT * FROM {transactions_table}""")
display(transactions)

In [0]:
repayments = spark.table(repayments_table)
display(repayments)

## 2. Basic Filtering
Same result, shown via both the PySpark API and the Spark SQL API.

In [0]:
# PySpark API - .where() method
late_repayments_py = repayments.where("payment_status = 'Late'")
display(late_repayments_py)

In [0]:
spark.sql(f"SELECT * FROM {catalog}.{schema}.repayments WHERE payment_status = 'Late'")

## 3. Basic Transformations (PySpark API)

In [0]:
# Importing required functions
# (wildcard - avoid in production code - import only what is needed)
from pyspark.sql.functions import *

In [0]:
# Selecting - Subset
subset_df = customers.select("customer_id", "product_type", "credit_limit")
display(subset_df)

In [0]:
# Selecting - All
all_columns_df = customers.select(["*"])
display(all_columns_df)

In [0]:
# Filtering - Specific Records
# (SAS Context: this is your WHERE clause or IF subsetting in a data step)
low_limit_df = (
    customers.select("customer_id", "product_type", "credit_limit")
    .where(col("credit_limit") < 5000)
)
display(low_limit_df)

## 4. Advanced Transformations: Distinct & Ordered Data (PySpark API)

In [0]:
# Collecting Distinct Records
distinct_products_df = customers.dropDuplicates(["product_type", "city"]).select("product_type", "city")
display(distinct_products_df)

In [0]:
# Ordering Records
# Sorting records in ascending order, use ascending=False for descending order
sorted_df = transactions.sort("amount", ascending=True)
display(sorted_df)

# The orderBy function can also be used
ordered_df = transactions.orderBy("amount", ascending=False)
display(ordered_df)

## 5. Advanced Transformations: GroupBy (PySpark API)

In [0]:
# Total transaction value by merchant category
category_totals_df = transactions.groupBy("merchant_category").sum("amount")
display(category_totals_df)

In [0]:
# Volume of transactions per customer
transaction_count_df = transactions.groupBy("customer_id").count()
display(transaction_count_df)

## 6. Advanced Transformations: Aggregation

In [0]:
# PySpark API - min amount_due by payment_status
min_due_df = repayments.groupBy("payment_status").min("amount_due")
display(min_due_df)

In [0]:
# groupBy and aggregate on multiple columns
merchant_totals_df = transactions.groupBy("merchant_category", "transaction_type").sum("amount")
display(merchant_totals_df)

In [0]:
# agg() - compute multiple aggregations in a single statement
customer_agg_df = (
    transactions.groupBy("customer_id")
    .agg(
        avg("amount").alias("avg_amount"),
        sum("amount").alias("sum_amount"),
        max("amount").alias("max_amount"),
        count("transaction_id").alias("total_transactions"),
    )
)
display(customer_agg_df)

In [0]:
# Spark SQL API - aggregations with GROUP BY
spark.sql(f""" SELECT
customer_id,
  avg(amount) AS avg_amount,
  sum(amount) AS sum_amount,
  max(amount) AS max_amount,
  count(transaction_id) AS total_transactions
FROM {catalog}.{schema}.transactions
GROUP BY customer_id""")

## 7. Advanced Transformations: Pivoting (PySpark API)

In [0]:
# Group by customer, pivot by merchant_category, and get the total sum of amount
pivot_df = transactions.groupBy("customer_id").pivot("merchant_category").sum("amount")
display(pivot_df)

## 8. Advanced Transformations: Joins (PySpark API)

In [0]:
# Inner Join - default join in PySpark
inner_join_df = transactions.join(customers, transactions.customer_id == customers.customer_id, "inner")
display(inner_join_df)

In [0]:
# Full Outer Join - returns all rows from both datasets, nulls where there is no match
full_outer_join_df = transactions.join(customers, transactions.customer_id == customers.customer_id, "outer")
display(full_outer_join_df)

In [0]:
# Left Join - all rows from the left dataset (transactions), nulls where no match on the right
left_outer_join_df = transactions.join(customers, transactions.customer_id == customers.customer_id, "left")
display(left_outer_join_df)

In [0]:
# Right Join - all rows from the right dataset (customers), nulls where no match on the left
right_outer_join_df = transactions.join(customers, transactions.customer_id == customers.customer_id, "right")
display(right_outer_join_df)

## 9. Advanced Transformations: Union & Intersect (PySpark API)

In [0]:
# Split customers into two subsets, then bring them back together
current_product_df = customers.where("product_type = 'Credit Card'")
savings_product_df = customers.where("product_type = 'Savings'")

# Keep duplicates in union
union_df = current_product_df.union(savings_product_df)
display(union_df)

# Use the distinct() method to remove any duplicates from the union
union_df_no_duplicates = current_product_df.union(savings_product_df).distinct()
display(union_df_no_duplicates)

# unionByName matches on column names (column ordering is irrelevant across both DataFrames)
union_by_name_df = current_product_df.unionByName(savings_product_df)
display(union_by_name_df)

In [0]:
# Intersect - common rows between two DataFrames
late_customer_ids = late_repayments_py.select("customer_id").distinct()
credit_card_customer_ids = current_product_df.select("customer_id").distinct()

intersect_df = late_customer_ids.intersect(credit_card_customer_ids)
display(intersect_df)

## 10. Testing

In [0]:
from pyspark.testing import assertDataFrameEqual

# Compare the PySpark filter result against a Spark SQL equivalent
late_repayments_sql = spark.sql(f"SELECT * FROM {repayments_table} WHERE payment_status = 'Late'")

assertDataFrameEqual(late_repayments_py, late_repayments_sql)

## 11. Writing to External Sources

In [0]:
print((f"{catalog}.{write_schema}.customer_transaction_summary"))

In [0]:
# Writing - overwrite mode
customer_agg_df.write.mode("overwrite").saveAsTable(f"{catalog}.{write_schema}.customer_transaction_summary")

In [0]:
# Writing - append mode
customer_agg_df.write.mode("append").saveAsTable(f"{catalog}.{write_schema}.customer_transaction_summary")

## 12. Optimisation

In [0]:
# Broadcast join - customers is the small reference table, transactions is the large fact table
# No shuffle of the large dataset is needed
broadcast_join_df = transactions.join(customers.hint("broadcast"), on="customer_id", how="left")
broadcast_join_df.show()